# Covariance validation paper: figures from the NERSC files

Every figure of the paper, computed directly from the original files at NERSC. Edit anything.

* **Run on NERSC Jupyter** (`jupyter.nersc.gov`) with the **DESI** kernel (cosmodesi `main`), or any kernel with numpy,
  scipy, matplotlib and healpy. `thecov` is imported from `~/thecov`.
* **Structure.** Section 0 sets paths, style and loads the data (run it once). Each figure then has two cells:
  *compute* (makes plain arrays with explicit names) and *plot* (pure matplotlib, re-run as often as you like).
  `savefig(fig, name)` writes `paper/figures/<name>.pdf`, which `make paper` uses.
* **Models.** `C['G_local']` Gaussian with the local window $m^2$; `C['G']` Gaussian with the $\xi$-kernel window;
  `C['rec']` = G + SSC + discreteness (recommended, nothing fitted); `terms[...]` the individual non-Gaussian terms.
* Data vectors are $(P_0, P_2, P_4)$ concatenated, 56 $k$-bins each ($\Delta k = 0.005$, $0.02<k<0.3$).

## 0. Paths, style, data

In [ ]:
import os
# ---- paths: the original files (edit if you moved them) ----
OUT       = '/global/cfs/cdirs/desicollab/users/oalves/thecov_validation'      # ssc_*, maps, n(z), hole statistics
HOME      = os.path.expanduser('~')                                           # /global/homes/o/oalves at NERSC
HOLI_DIR  = HOME + '/thecov_desi/holi_v3_mock173'              # report_data_holi-*.npz (Gaussian dumps)
ABACUS_COMPLETE_DIR = HOME + '/thecov_desi/abacus-2ndgen-dr2-complete_mock0'
ABACUS_ALTMTL_DIR   = HOME + '/thecov_desi/abacus-2ndgen-dr2-altmtl_mock0'
FIGDIR    = HOME + '/thecov/paper/figures'                     # where savefig() writes

FILES = {
    'holi_gauss':   HOLI_DIR + '/report_data_holi-kcore2-{tracer}.npz',      # mocks + Gaussian cov, xi-kernel window
    'holi_local':   HOLI_DIR + '/report_data_holi-altmtl.npz',               # same mocks, local window m^2 / own weights
    'holi_ng':      OUT + '/ssc_holi-kcore2-{tracer}.npz',                   # non-Gaussian terms (ssc_check)
    'abacus_complete': ABACUS_COMPLETE_DIR + '/report_data_abacus-complete.npz',
    'abacus_altmtl':   ABACUS_ALTMTL_DIR + '/report_data_abacus-altmtl.npz',
    'abacus_gauss': ABACUS_COMPLETE_DIR + '/report_data_abacus-kcore-complete-LRG1.npz',
    'abacus_ng':    OUT + '/ssc_abacus-kcore-complete-LRG1.npz',
    'maps':         OUT + '/paper_maps_{tracer}.npz',
    'holes':        OUT + '/hole_fraction_{tracer}.json',
    'nz':           OUT + '/nz_scatter_{tracer}.npz',
}

import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, HOME + '/thecov')
from thecov.covariance_tools import window_convolve
%matplotlib inline
os.makedirs(FIGDIR, exist_ok=True)
for name, f in FILES.items():
    for t in ('LRG1', 'QSO'):
        p = f.format(tracer=t)
        if not os.path.exists(p):
            print('missing:', p)

In [ ]:
# ---- style: change freely; one colour per model and per multipole, used by every figure ----
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = ('#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4',
                                                          '#008300', '#4a3aa7', '#e34948')
GREY, LIGHT = '#898781', '#e1e0d9'
ELL_COLOR   = {0: BLUE, 2: ORANGE, 4: VIOLET}
MODEL_COLOR = {'G_local': GREY, 'G': BLUE, 'G+SSC': ORANGE, 'rec': AQUA, 'rec+T0tree': RED, 'rec+T0resp': VIOLET}
MODEL_LABEL = {'G_local': r'Gaussian, local window $m^2$', 'G': r'Gaussian, $\xi$-kernel window',
               'G+SSC': 'Gaussian + SSC', 'rec': 'Gaussian + SSC + discreteness',
               'rec+T0tree': r'... + tree-level $T_0$', 'rec+T0resp': r'... + response $T_0$'}
DIVERGING  = LinearSegmentedColormap.from_list('div', ['#184f95', '#6da7ec', '#f0efec', '#ec835a', '#b8322f'])
SEQUENTIAL = LinearSegmentedColormap.from_list('seq', ['#f0efec', '#9ec5f4', '#3987e5', '#184f95', '#0d366b'])
WIDTH = 6.3                     # inches: JCAP text width
KLABEL = r'$k\ [h\,{\rm Mpc}^{-1}]$'

plt.rcParams.update({'font.size': 8.5, 'legend.fontsize': 7.5, 'xtick.labelsize': 7.5, 'ytick.labelsize': 7.5,
                     'font.family': 'serif', 'mathtext.fontset': 'cm', 'axes.grid': True, 'grid.color': LIGHT,
                     'grid.linewidth': 0.5, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titlelocation': 'left', 'axes.titlesize': 8.5, 'legend.frameon': False,
                     'lines.linewidth': 1.4, 'figure.dpi': 120, 'savefig.bbox': 'tight'})

def savefig(fig, name):
    fn = f'{FIGDIR}/{name}.pdf'
    fig.savefig(fn)
    print('wrote', fn)

In [ ]:
# ---- defective mocks: n(z) bins more than 6 robust sigma from the median (12 LRG mocks with +55-65% in 0.46<z<0.47) ----
EXCLUDED = {}
for tracer in ('LRG1', 'QSO'):
    z = np.load(FILES['nz'].format(tracer=tracer))
    bad = set()
    for cap in ('NGC', 'SGC'):
        dev = z[f'{cap}/NZ'] / np.median(z[f'{cap}/NZ'], axis=0) - 1
        mad = 1.4826 * np.median(np.abs(dev - np.median(dev, axis=0)), axis=0)
        outlier = (np.abs(dev - np.median(dev, axis=0)) > 6 * mad).any(axis=1)
        bad |= set(z[f'{cap}/mocks'][outlier].tolist())
    EXCLUDED[tracer] = sorted(bad)
    print(tracer, 'excluded mocks:', EXCLUDED[tracer])

In [ ]:
# ---- data loaders: everything a figure needs for one tracer / cap / bin width ----
from functools import lru_cache
NB = 56                          # k bins per multipole at Delta k = 0.005
ELLS = (0, 2, 4)

def block(ell, nb=NB):
    # slice of multipole ell in a concatenated (P0, P2, P4) vector
    j = ELLS.index(ell)
    return slice(j * nb, (j + 1) * nb)

def rebin_matrix(nmodes, f):
    # R @ P_fine = mode-weighted average over groups of f bins (exact for the measured vectors)
    nb = len(nmodes)
    R1 = np.zeros((nb // f, nb))
    for i in range(nb // f):
        w = nmodes[i * f:(i + 1) * f]
        R1[i, i * f:(i + 1) * f] = w / w.sum()
    return np.kron(np.eye(3), R1)

@lru_cache(None)
def holi(tracer, cap, rebin=1, keep_excluded=False):
    # returns dict: k, V (mocks x 168), ids, C (dict of model covariances), terms (dict of non-Gaussian terms), params, plin
    g = np.load(FILES['holi_gauss'].format(tracer=tracer))
    loc = np.load(FILES['holi_local'])
    p = f'{tracer}/{cap}/x{rebin}'
    ids = np.array([int(s[4:]) for s in g[p + '/mock_ids']])
    keep = np.ones(len(ids), bool) if keep_excluded else ~np.isin(ids, EXCLUDED[tracer])
    d = dict(k=g[p + '/k'], V=g[p + '/V'][keep].astype(float), ids=ids[keep], nmodes=g[p + '/nmodes'])
    d['C'] = {'G': g[f'{p}/C/' + ('combined-regions' if cap == 'GCcomb' else 'kernel')],
              'G_local': loc[f'{p}/C/' + ('combined-regions' if cap == 'GCcomb' else 'random-density')]}
    if f'{p}/C/none' in loc.files:
        d['C']['G_ownweight'] = loc[f'{p}/C/none']
    # non-Gaussian terms are saved at Delta k = 0.005; coarser bins: R T R^T
    ng = np.load(FILES['holi_ng'].format(tracer=tracer))
    R = np.eye(3 * NB) if rebin == 1 else rebin_matrix(g[f'{tracer}/{cap}/x1/nmodes'], rebin)
    terms = {}
    for name, key in (('ssc', 'C_ssc'), ('ssc_noLA', 'C_ssc_noLA'), ('disc', 'C_disc'), ('T0_response', 'C_T0_response')):
        if f'{cap}/{key}' in ng.files:
            terms[name] = R @ ng[f'{cap}/{key}'] @ R.T
    if f'{cap}/C_T0_snake' in ng.files:          # tree-level T0, window-convolved as in ssc_check
        Cg1 = g[f'{tracer}/{cap}/x1/C/kernel']
        terms['T0_tree'] = R @ window_convolve(ng[f'{cap}/C_T0_snake'] + ng[f'{cap}/C_T0_star'], Cg1) @ R.T
    d['terms'] = terms
    d['C']['G+SSC'] = d['C']['G'] + terms['ssc']
    d['C']['rec'] = d['C']['G+SSC'] + terms['disc']
    if 'T0_tree' in terms:
        d['C']['rec+T0tree'] = d['C']['rec'] + terms['T0_tree']
    if 'T0_response' in terms:
        d['C']['rec+T0resp'] = d['C']['rec'] + terms['T0_response']
    pc = cap if cap != 'GCcomb' else 'NGC'
    d['params'] = dict(zip(('b1', 'b2', 'bs2', 'f', 'zeff'), ng[f'{pc}/params']))
    d['plin'] = (ng[f'{pc}/plin_k'], ng[f'{pc}/plin_P_damped_normalised'])
    return d

@lru_cache(None)
def abacus(kind, cap, rebin=1):
    # kind: 'complete' / 'altmtl' (local-window Gaussian), or 'kernel' (complete, xi-kernel window + non-Gaussian terms)
    if kind in ('complete', 'altmtl'):
        z = np.load(FILES['abacus_' + kind])
        p = f'LRG1/{cap}/x{rebin}'
        return dict(k=z[p + '/k'], V=z[p + '/V'].astype(float), ids=z[p + '/mock_ids'],
                    C={'G_local': z[f'{p}/C/' + ('combined-regions' if cap == 'GCcomb' else 'random-density')]})
    g, ng = np.load(FILES['abacus_gauss']), np.load(FILES['abacus_ng'])
    p = f'LRG1/{cap}/x1'
    CG = g[f'{p}/C/' + ('combined-regions' if cap == 'GCcomb' else 'kernel')]
    terms = {'ssc': ng[f'{cap}/C_ssc'], 'disc': ng[f'{cap}/C_disc']}
    if f'{cap}/C_T0_response' in ng.files:
        terms['T0_response'] = ng[f'{cap}/C_T0_response']
    if f'{cap}/C_T0_snake' in ng.files:
        terms['T0_tree'] = window_convolve(ng[f'{cap}/C_T0_snake'] + ng[f'{cap}/C_T0_star'], CG)
    return dict(k=g[p + '/k'], V=g[p + '/V'].astype(float), C={'G': CG}, terms=terms)

d = holi('LRG1', 'NGC')
print('LRG1 NGC:', d['V'].shape, 'mocks x bins;  models:', list(d['C']), ';  terms:', list(d['terms']))

In [ ]:
# ---- statistics used by several figures ----
def corr(C):
    s = np.sqrt(np.diag(C))
    return C / np.outer(s, s)

def kidx(k, kmax, ells=ELLS):
    # indices of the bins with k <= kmax, in every multipole
    nb = len(k)
    s = np.flatnonzero(k <= kmax + 1e-9)
    return np.concatenate([s + ELLS.index(l) * nb for l in ells])

def chi2(V, C, idx=None):
    # chi^2 of every mock about the mock mean (expectation n (N-1)/N)
    idx = np.arange(V.shape[1]) if idx is None else idx
    L = np.linalg.cholesky(C[np.ix_(idx, idx)])
    x = np.linalg.solve(L, (V[:, idx] - V[:, idx].mean(0)).T)
    return (x ** 2).sum(0)

def whiten(V, C):
    # returns (C^-1/2 S C^-1/2, C^-1/2)
    w, U = np.linalg.eigh(C)
    Wi = (U / np.sqrt(w)) @ U.T
    return Wi @ np.cov(V.T) @ Wi, Wi

def wishart_sigma(C, N):
    # std of the sample-covariance elements for N Gaussian vectors with covariance C
    d = np.diag(C)
    return np.sqrt((np.outer(d, d) + C ** 2) / (N - 1))

def fit_templates(V, CG, templates):
    # Wishart ML amplitudes of C = CG + sum theta_a T_a, with Fisher errors
    from scipy.optimize import minimize
    S, N, T = np.cov(V.T), len(V), list(templates.values())
    def m2lnL(th):
        try:
            L = np.linalg.cholesky(CG + sum(t * Ta for t, Ta in zip(th, T)))
        except np.linalg.LinAlgError:
            return 1e300
        X = np.linalg.solve(L, S)
        return (N - 1) * (np.trace(np.linalg.solve(L, X.T)) + 2 * np.log(np.diag(L)).sum())
    x0 = np.ones(len(T)) if m2lnL(np.ones(len(T))) < 1e299 else 0.1 * np.ones(len(T))
    th = minimize(m2lnL, x0, method='Powell', options=dict(xtol=1e-4, ftol=1e-6, maxiter=20000)).x
    Ci = np.linalg.inv(CG + sum(t * Ta for t, Ta in zip(th, T)))
    A = [Ci @ Ta for Ta in T]
    F = np.array([[0.5 * (N - 1) * np.trace(a @ b) for b in A] for a in A])
    return dict(zip(templates, th)), dict(zip(templates, np.sqrt(np.diag(np.linalg.inv(F)))))

## 1. The window: $\xi$-kernel and weights (`window`)
(a) the kernel $K_k(r)=\xi_0(r)j_0(kr)/P_0(k)$ that pair-averages the window; (b) Gaussian variance with the $\xi$-kernel
window over that with the local window $m^2$; (c) own weight per random over local mean weight.

In [ ]:
from scipy.special import spherical_jn
from scipy.integrate import simpson
d = holi('LRG1', 'NGC')
b1, f = d['params']['b1'], d['params']['f']
kl, Pl = d['plin']
P0 = (b1**2 + 2/3 * b1 * f + f**2 / 5) * Pl                      # Kaiser monopole
kk = np.linspace(1e-4, 3.0, 30000)
pk = np.interp(kk, kl, P0) * np.exp(-kk**2)                        # damped to regularise xi at small r
r = np.linspace(0.5, 150, 600)
xi0 = np.array([simpson(kk**2 * pk * spherical_jn(0, kk * ri), x=kk) for ri in r]) / (2 * np.pi**2)
kernel = {kv: 4 * np.pi * r**2 * xi0 * spherical_jn(0, kv * r) / np.interp(kv, kl, P0) for kv in (0.05, 0.1, 0.2)}

ratio_kernel, ratio_own = {}, {}                                 # [(tracer, cap)] -> diag ratio vector
for tracer in ('LRG1', 'QSO'):
    for cap in ('NGC', 'SGC'):
        d = holi(tracer, cap)
        ratio_kernel[tracer, cap] = np.diag(d['C']['G']) / np.diag(d['C']['G_local'])
        ratio_own[tracer, cap] = np.diag(d['C']['G_ownweight']) / np.diag(d['C']['G_local'])
k = d['k']

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(WIDTH, 2.3), gridspec_kw=dict(width_ratios=[1.1, 1, 1]))
for (kv, y), c in zip(kernel.items(), (BLUE, ORANGE, VIOLET)):
    ax[0].plot(r, y, color=c, label=f'$k={kv}$')
ax[0].set(xlabel=r'$r\ [h^{-1}{\rm Mpc}]$', ylabel=r'$4\pi r^2 K_k(r)$', title=r'(a) $\xi$-kernel', xlim=(0, 150))
ax[0].legend()
for a, ratios, title in ((ax[1], ratio_kernel, r'(b) $\xi$-kernel / local window'), (ax[2], ratio_own, '(c) own weight / mean weight')):
    for (tracer, cap), y in ratios.items():
        for ell, c in ELL_COLOR.items():
            a.plot(k, y[block(ell)], color=c, ls='-' if cap == 'NGC' else '--', lw=1.4 if tracer == 'LRG1' else 0.8,
                   label=f'$\\ell={ell}$' if (tracer, cap) == ('LRG1', 'NGC') else None)
    a.axhline(1, color='k', lw=0.6)
    a.set(xlabel=KLABEL, title=title, ylim=(0.985, 1.13))
ax[1].set_ylabel('ratio of Gaussian variances')
ax[1].legend()
fig.tight_layout()
savefig(fig, 'window')

## 2. Size of each term on the diagonal (`budget`)
$C^X_{ii}/C^{\rm G}_{ii}$ per multipole; translucent: negative values (absolute value), where the local-average term
makes the SSC monopole diagonal negative.

In [ ]:
BUDGET_TERMS = [('ssc', 'SSC', ORANGE, '-'), ('ssc_noLA', 'SSC without local average', ORANGE, ':'),
                ('disc', 'discreteness', AQUA, '-'), ('T0_tree', r'tree $T_0$ (rejected)', RED, '--'),
                ('T0_response', r'response $T_0$ (rejected)', VIOLET, ':')]
budget = {}                                                      # [(tracer, term)] -> diag(C_term)/diag(C_G)
for tracer in ('LRG1', 'QSO'):
    d = holi(tracer, 'NGC')
    for key, *_ in BUDGET_TERMS:
        if key in d['terms']:
            budget[tracer, key] = np.diag(d['terms'][key]) / np.diag(d['C']['G'])
k = d['k']

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(WIDTH, 3.7), sharex=True, sharey=True)
for row, tracer in enumerate(('LRG1', 'QSO')):
    for col, ell in enumerate(ELLS):
        a = ax[row, col]
        for key, label, c, ls in BUDGET_TERMS:
            if (tracer, key) not in budget:
                continue
            y = budget[tracer, key][block(ell)]
            a.plot(k, np.where(y > 0, y, np.nan), color=c, ls=ls, label=label)
            a.plot(k, np.where(y < 0, -y, np.nan), color=c, lw=3, alpha=0.25)
        a.set_yscale('log'); a.set_ylim(1e-4, 1)
        if row == 0: a.set_title(f'$\\ell={ell}$')
        if col == 0: a.set_ylabel(f'{tracer} NGC\n' + r'$C^X_{ii}/C^{\rm G}_{ii}$')
        if row == 1: a.set_xlabel(KLABEL)
fig.legend(*ax[0, 0].get_legend_handles_labels(), loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.06))
fig.tight_layout()
savefig(fig, 'budget')

## 3. Structure of each term (`anatomy`)
(a) eigenvalues of $C_{\rm G}^{-1/2}C_XC_{\rm G}^{-1/2}$ (how many directions a term acts on); (b, c) the two leading
directions of SSC and discreteness, as patterns in the data vector in units of the Gaussian error per bin.

In [ ]:
d = holi('LRG1', 'NGC')
w, U = np.linalg.eigh(d['C']['G'])
G_half, G_mhalf = (U * np.sqrt(w)) @ U.T, (U / np.sqrt(w)) @ U.T
sigma_G = np.sqrt(np.diag(d['C']['G']))
eigvals, patterns = {}, {}                                       # patterns[term] = list of 2 data-vector patterns
for key in ('ssc', 'disc', 'T0_tree', 'T0_response'):
    lam, vec = np.linalg.eigh(G_mhalf @ d['terms'][key] @ G_mhalf)
    order = np.argsort(-np.abs(lam))
    lam, vec = lam[order], vec[:, order]
    eigvals[key] = np.abs(lam)
    pats = []
    for m in range(2):
        p = G_half @ vec[:, m] * np.sqrt(abs(lam[m])) / sigma_G
        pats.append(p * np.sign(p[np.argmax(np.abs(p))]))
    patterns[key] = pats
k = d['k']

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(WIDTH, 2.4), gridspec_kw=dict(width_ratios=[1, 1.25, 1.25]))
for key, label, c in (('ssc', 'SSC', ORANGE), ('disc', 'discreteness', AQUA), ('T0_tree', r'tree $T_0$', RED),
                      ('T0_response', r'response $T_0$', VIOLET)):
    ax[0].plot(np.arange(1, 31), eigvals[key][:30], 'o-', ms=2.5, lw=1, color=c, label=label)
ax[0].set(yscale='log', ylim=(1e-4, 30), xlabel='rank', title='(a) eigenvalues')
ax[0].legend(loc='lower right', fontsize=6.5)
for a, key, title in ((ax[1], 'ssc', '(b) SSC'), (ax[2], 'disc', '(c) discreteness')):
    for p, ls in zip(patterns[key], ('-', '--')):
        for ell, c in ELL_COLOR.items():
            a.plot(k, p[block(ell)], color=c, ls=ls, lw=1.1, label=f'$\\ell={ell}$' if ls == '-' else None)
    a.axhline(0, color='k', lw=0.5)
    a.set(xlabel=KLABEL, title=title + ': leading directions')
ax[1].set_ylabel(r'$\sqrt{\lambda}\,v/\sigma_{\rm G}$'); ax[1].legend(fontsize=6.5)
fig.tight_layout()
savefig(fig, 'anatomy')

## 4. Diagonals (`diagonals_LRG1`, `diagonals_QSO`)
Mock variance over model variance per multipole, averaged over groups of `GROUP` bins; error bars: Wishart scatter of
that average under the model (includes the correlation of neighbouring bins).

In [ ]:
GROUP = 4
DIAG_MODELS = ('G_local', 'G', 'rec')
diag = {}       # [(tracer, cap, model, ell)] -> (k_centres, ratio, error)
for tracer in ('LRG1', 'QSO'):
    for cap in ('NGC', 'SGC'):
        d = holi(tracer, cap)
        V, k, N = d['V'], d['k'], len(d['V'])
        S = np.cov(V.T)
        for m in DIAG_MODELS:
            C = d['C'][m]
            ratio, rho2 = np.diag(S) / np.diag(C), corr(C)**2
            for ell in ELLS:
                kc, y, e = [], [], []
                for g in range(NB // GROUP):
                    i = block(ell).start + g * GROUP + np.arange(GROUP)
                    kc.append(k[i - block(ell).start].mean()); y.append(ratio[i].mean())
                    e.append(np.sqrt(2 * rho2[np.ix_(i, i)].sum() / (N - 1)) / GROUP)
                diag[tracer, cap, m, ell] = np.array(kc), np.array(y), np.array(e)

In [ ]:
for tracer in ('LRG1', 'QSO'):
    fig, ax = plt.subplots(2, 3, figsize=(WIDTH, 3.6), sharex=True, sharey=True)
    for row, cap in enumerate(('NGC', 'SGC')):
        for col, ell in enumerate(ELLS):
            a = ax[row, col]
            for j, m in enumerate(DIAG_MODELS):
                kc, y, e = diag[tracer, cap, m, ell]
                a.errorbar(kc + (j - 1) * 0.0025, y, e, fmt='o', ms=3, lw=0.9, color=MODEL_COLOR[m], label=MODEL_LABEL[m])
            a.axhline(1, color='k', lw=0.6)
            if row == 0: a.set_title(f'$\\ell={ell}$')
            if col == 0: a.set_ylabel(f'{tracer} {cap}\n' + r'$\hat\sigma^2/\sigma^2_{\rm model}$')
            if row == 1: a.set_xlabel(KLABEL)
    ax[0, 0].set_ylim(0.82, 1.32)
    fig.legend(*ax[0, 0].get_legend_handles_labels(), loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.05))
    fig.tight_layout()
    savefig(fig, f'diagonals_{tracer}')

## 5. Veto holes (`holes`)
(a) fill fraction of the randoms at nside 256; (b) zoom at nside `ZOOM_NSIDE`; (c) $R_{PP}=\langle f^2\rangle/\langle f\rangle^2$
vs scale (the extra Gaussian variance the hole-fraction inhomogeneity can add relative to the local window).

In [ ]:
import healpy as hp
HOLES_TRACER, HOLES_CAP, ZOOM_NSIDE = 'LRG1', 'NGC', 1024
z = np.load(FILES['maps'].format(tracer=HOLES_TRACER))
E = float(z[f'{HOLES_CAP}/E'])                                   # expected randoms in a full nside-2048 pixel

def fill_map(nside, pix, counts):
    m = np.full(hp.nside2npix(nside), hp.UNSEEN)
    m[pix] = counts / (E * (2048 // nside)**2)
    return hp.reorder(m, n2r=True)

m256 = fill_map(256, z[f'{HOLES_CAP}/pix256'], z[f'{HOLES_CAP}/n256'])
lon, lat = hp.pix2ang(256, np.flatnonzero(m256 != hp.UNSEEN), lonlat=True)
ra_c = np.degrees(np.angle(np.mean(np.exp(1j * np.radians(lon))))) % 360
img_full = hp.projector.CartesianProj(rot=(ra_c, 0), lonra=[-60, 60], latra=[-15, 85], xsize=900).projmap(
    m256, lambda x, y, zz: hp.vec2pix(256, x, y, zz))
shift = 2 * int(np.log2(2048 // ZOOM_NSIDE))
zp, zi = np.unique(z[f'{HOLES_CAP}/zoom_pix'] >> shift, return_inverse=True)
mz = fill_map(ZOOM_NSIDE, zp, np.bincount(zi, weights=z[f'{HOLES_CAP}/zoom_n']))
ra0, dec0 = z[f'{HOLES_CAP}/zoom_center']
img_zoom = hp.projector.GnomonicProj(rot=(ra0, dec0), xsize=500, reso=4 * 60 / 500).projmap(
    mz, lambda x, y, zz: hp.vec2pix(ZOOM_NSIDE, x, y, zz))
rpp = {}                                                         # [(tracer, cap)] -> (scale_deg, R_PP)
for tracer in ('LRG1', 'QSO'):
    js = json.load(open(FILES['holes'].format(tracer=tracer)))
    for cap in ('NGC', 'SGC'):
        st = js[cap]['stats']['local-mean-weight']
        rpp[tracer, cap] = np.array([s['scale_arcmin'] / 60 for s in st]), np.array([s['R_PP'] for s in st])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(WIDTH, 2.5), gridspec_kw=dict(width_ratios=[1.5, 1, 1.05], wspace=0.6))
im = ax[0].imshow(np.ma.masked_less(img_full, -1e30), origin='lower', extent=(ra_c + 60, ra_c - 60, -15, 85),
                  cmap=SEQUENTIAL, vmin=0.6, vmax=1.0, aspect='auto', interpolation='nearest')
ax[0].add_patch(plt.Rectangle((ra0 - 2, dec0 - 2), 4, 4, fill=False, color=RED))
ax[0].set(xlabel='RA [deg]', ylabel='Dec [deg]', title='(a) fill fraction, nside 256'); ax[0].grid(False)
fig.colorbar(im, ax=ax[0], fraction=0.05, pad=0.02)
ax[1].imshow(np.ma.masked_less(img_zoom, -1e30), origin='lower', extent=(2, -2, -2, 2), cmap=SEQUENTIAL,
             vmin=0, vmax=1.2, interpolation='nearest')
ax[1].set(xlabel=r'$\Delta$RA [deg]', ylabel=r'$\Delta$Dec [deg]', title=f'(b) zoom, nside {ZOOM_NSIDE}'); ax[1].grid(False)
for (tracer, cap), (s, R) in rpp.items():
    ax[2].plot(s, R, marker='o', ms=2.5, color=BLUE if tracer == 'LRG1' else ORANGE, ls='-' if cap == 'NGC' else '--',
               label=f'{tracer} {cap}')
ax[2].set(xscale='log', xlabel='pixel scale [deg]', ylabel=r'$R_{PP}$', title='(c) hole-fraction variance')
ax[2].axhline(1, color='k', lw=0.6); ax[2].legend(fontsize=6.3)
savefig(fig, 'holes')

## 6. Correlation matrix and residual maps (`corr`)
(a) correlation of the mocks (upper left) and of the recommended model (lower right); (b, c) $(S-C)/\sigma_W$ with
$\sigma_W$ the Wishart scatter under the model: Gaussian (upper left) and recommended (lower right).

In [ ]:
def split(upper_left, lower_right):
    # one image: upper_left above the diagonal, lower_right below (with origin='lower')
    M = lower_right.copy()
    il = np.tril_indices(len(M), -1)
    M[il] = upper_left[il]
    np.fill_diagonal(M, np.nan)
    return M

d = holi('LRG1', 'NGC')
corr_img = split(corr(np.cov(d['V'].T)), corr(d['C']['rec']))
resid_img = {}
for tracer in ('LRG1', 'QSO'):
    d = holi(tracer, 'NGC')
    S, N = np.cov(d['V'].T), len(d['V'])
    R = {m: (S - d['C'][m]) / wishart_sigma(d['C'][m], N) for m in ('G', 'rec')}
    resid_img[tracer] = split(R['G'], R['rec'])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(WIDTH, 2.45))
panels = [(corr_img, 0.6, '(a) LRG1 NGC correlation', ('mocks', 'model')),
          (resid_img['LRG1'], 4, r'(b) LRG1 NGC, $(S-C)/\sigma_W$', ('Gaussian', 'recommended')),
          (resid_img['QSO'], 4, r'(c) QSO NGC, $(S-C)/\sigma_W$', ('Gaussian', 'recommended'))]
for a, (img, vmax, title, (ul, lr)) in zip(ax, panels):
    im = a.imshow(img, cmap=DIVERGING, vmin=-vmax, vmax=vmax, origin='lower', interpolation='nearest')
    for j in (1, 2):
        a.axhline(j * NB - 0.5, color='k', lw=0.4); a.axvline(j * NB - 0.5, color='k', lw=0.4)
    t = [j * NB + NB / 2 for j in range(3)]
    a.set_xticks(t, [r'$\ell=0$', r'$\ell=2$', r'$\ell=4$']); a.set_yticks(t, [r'$\ell=0$', r'$\ell=2$', r'$\ell=4$'], rotation=90)
    a.tick_params(length=0); a.grid(False); a.set_title(title, fontsize=8)
    a.text(0.03, 0.97, ul, transform=a.transAxes, va='top', fontsize=6.8, bbox=dict(fc='w', ec='none', alpha=0.75))
    a.text(0.97, 0.03, lr, transform=a.transAxes, ha='right', fontsize=6.8, bbox=dict(fc='w', ec='none', alpha=0.75))
    fig.colorbar(im, ax=a, fraction=0.046, pad=0.03)
fig.tight_layout()
savefig(fig, 'corr')

## 7. $\chi^2$ (`chi2`)
(a) $\chi^2$ of each mock (LRG1 NGC, $k<0.3$) vs $\chi^2_n$; (b, c) $\langle\chi^2\rangle/n$ vs $k_{\max}$.

In [ ]:
from scipy.stats import chi2 as chi2_dist
CHI2_MODELS = ('G_local', 'G', 'rec')
d = holi('LRG1', 'NGC')
N, n = len(d['V']), d['V'].shape[1]
chi2_mocks = {m: chi2(d['V'], d['C'][m]) * N / (N - 1) for m in CHI2_MODELS}     # rescaled to n dof
KMAX = np.arange(0.06, 0.301, 0.02)
chi2_kmax = {}                                                   # [(tracer, cap, model)] -> <chi2>/n per kmax
for tracer in ('LRG1', 'QSO'):
    for cap in ('NGC', 'SGC'):
        d = holi(tracer, cap)
        for m in CHI2_MODELS:
            chi2_kmax[tracer, cap, m] = np.array([chi2(d['V'], d['C'][m], kidx(d['k'], km)).mean()
                                                  / (len(kidx(d['k'], km)) * (len(d['V']) - 1) / len(d['V'])) for km in KMAX])
n_kmax = np.array([len(kidx(d['k'], km)) for km in KMAX])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(WIDTH, 2.3), gridspec_kw=dict(width_ratios=[1.15, 1, 1]))
bins = np.linspace(110, 330, 45)
for m in CHI2_MODELS:
    ax[0].hist(chi2_mocks[m], bins=bins, histtype='step', color=MODEL_COLOR[m], lw=1.2, label=MODEL_LABEL[m])
x = np.linspace(110, 330, 300)
ax[0].plot(x, chi2_dist.pdf(x, n) * N * (bins[1] - bins[0]), 'k--', lw=0.9, label=f'$\\chi^2_{{{n}}}$')
ax[0].set(xlabel=r'$\chi^2$ (LRG1 NGC)', ylabel='mocks per bin', title='(a)')
for a, tracer in zip(ax[1:], ('LRG1', 'QSO')):
    for cap, ls in (('NGC', '-'), ('SGC', '--')):
        for m in CHI2_MODELS:
            a.plot(KMAX, chi2_kmax[tracer, cap, m], color=MODEL_COLOR[m], ls=ls)
    a.fill_between(KMAX, 1 - np.sqrt(2 / (n_kmax * N)), 1 + np.sqrt(2 / (n_kmax * N)), color=LIGHT, lw=0)
    a.axhline(1, color='k', lw=0.6)
    a.set(xlabel=r'$k_{\max}$', ylim=(0.95, 1.2), title=f'{tracer}: ' + r'$\langle\chi^2\rangle/n$ (solid NGC, dashed SGC)')
fig.legend(*ax[0].get_legend_handles_labels(), loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.08), fontsize=7)
fig.tight_layout()
savefig(fig, 'chi2')

## 8. Whitened eigenvalue spectra (`eigen`)
Sorted eigenvalues of $C^{-1/2}SC^{-1/2}$ over the median of `N_NULL` Gaussian ensembles drawn from the model (band: their 95%).

In [ ]:
N_NULL = 200
rng = np.random.default_rng(1)
eig_ratio, null_band = {}, {}
for tracer in ('LRG1', 'QSO'):
    d = holi(tracer, 'NGC')
    N, n = d['V'].shape
    null = np.array([np.sort(np.linalg.eigvalsh(np.cov(rng.standard_normal((N, n)).T)))[::-1] for _ in range(N_NULL)])
    lo, med, hi = np.percentile(null, [2.5, 50, 97.5], axis=0)
    null_band[tracer] = lo / med, hi / med
    for m in ('G_local', 'G', 'rec'):
        W, _ = whiten(d['V'], d['C'][m])
        eig_ratio[tracer, m] = np.sort(np.linalg.eigvalsh(W))[::-1] / med

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(WIDTH, 2.4), sharey=True)
for a, tracer in zip(ax, ('LRG1', 'QSO')):
    rank = np.arange(1, len(null_band[tracer][0]) + 1)
    a.fill_between(rank, *null_band[tracer], color=LIGHT, lw=0, label='null, 95%')
    for m in ('G_local', 'G', 'rec'):
        a.plot(rank, eig_ratio[tracer, m], color=MODEL_COLOR[m], label=MODEL_LABEL[m])
    a.axhline(1, color='k', lw=0.6)
    a.set(xscale='log', yscale='log', xlabel='rank', title=f'{tracer} NGC')
ax[0].set_ylabel(r'$\lambda_i / \lambda_i^{\rm null}$'); ax[0].set_ylim(0.85, 3.2)
ax[0].set_yticks([0.9, 1, 1.5, 2, 3], ['0.9', '1', '1.5', '2', '3']); ax[0].minorticks_off()
ax[1].legend(fontsize=6.5)
fig.tight_layout()
savefig(fig, 'eigen')

## 9. Whitened clouds (`clouds`)
Mocks split in halves A/B: directions = leading eigenvectors of $C^{-1/2}S_AC^{-1/2}$; clouds = half B projected on them
(unit variance if the model is right). (c) a-priori directions: amplitudes of $P_0$ and $P_2$. (d) out-of-sample variance
along A's leading directions vs the (biased) in-sample eigenvalues. (e) the leading directions of the Gaussian model.

In [ ]:
SEED, N_DIR = 7, 20
d = holi('LRG1', 'NGC')
V, k = d['V'], d['k']
perm = np.random.default_rng(SEED).permutation(len(V))
A, B = V[perm[:len(V) // 2]], V[perm[len(V) // 2:]]
cv = {}                       # [model] -> dict(insample eigenvalues, directions U, B projected y, whitening Wi)
for m in ('G', 'rec'):
    W, Wi = whiten(A, d['C'][m])
    lam, U = np.linalg.eigh(W)
    o = np.argsort(-lam)
    cv[m] = dict(lam=lam[o], U=U[:, o], Wi=Wi, y=(Wi @ (B - B.mean(0)).T).T @ U[:, o])
# a-priori directions: GLS amplitudes of P0 and P2 under the recommended model
C = d['C']['rec']; Ci = np.linalg.inv(C); mu = V.mean(0)
T = np.zeros((len(mu), 2)); T[block(0), 0] = mu[block(0)]; T[block(2), 1] = mu[block(2)]
Finv = np.linalg.inv(T.T @ Ci @ T)
amp = (Finv @ T.T @ Ci @ (V - mu).T).T / np.sqrt(np.diag(Finv))
amp_model_cov = corr(Finv)
# patterns of the Gaussian model's leading directions in the data vector
Wh, sg = np.linalg.inv(cv['G']['Wi']), np.sqrt(np.diag(d['C']['G']))
lead_patterns = []
for i in range(2):
    p = Wh @ cv['G']['U'][:, i] / sg
    lead_patterns.append(p * np.sign(p[np.argmax(np.abs(p))]))
print('out-of-sample variances (G):', np.var(cv['G']['y'][:, :2], axis=0, ddof=1).round(2),
      ' (rec):', np.var(cv['rec']['y'][:, :2], axis=0, ddof=1).round(2))

In [ ]:
from matplotlib.patches import Ellipse, Circle
def ellipse(a, cov2, **kw):
    w, v = np.linalg.eigh(cov2)
    a.add_patch(Ellipse((0, 0), 2 * np.sqrt(w[1]), 2 * np.sqrt(w[0]), angle=np.degrees(np.arctan2(v[1, 1], v[0, 1])), fill=False, **kw))

def cloud(a, y, title, model_cov=None):
    a.scatter(y[:, 0], y[:, 1], s=2.5, color='0.35', alpha=0.55, lw=0)
    for r in (1, 2):
        if model_cov is None: a.add_patch(Circle((0, 0), r, fill=False, color=BLUE))
        else: ellipse(a, r**2 * model_cov, color=BLUE, ls='--')
    ellipse(a, np.cov(y.T), color=RED, lw=1.4)
    s = np.var(y, axis=0, ddof=1)
    a.text(0.03, 0.97, rf'$\hat\sigma^2={s[0]:.2f},\ {s[1]:.2f}$', transform=a.transAxes, va='top', fontsize=6.8)
    a.set(xlim=(-4.5, 4.5), ylim=(-4.5, 4.5), aspect='equal', title=title)

fig = plt.figure(figsize=(WIDTH, 4.5))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 0.85], hspace=0.45, wspace=0.35)
cloud(fig.add_subplot(gs[0, 0]), cv['G']['y'][:, :2], '(a) Gaussian')
cloud(fig.add_subplot(gs[0, 1]), cv['rec']['y'][:, :2], '(b) recommended')
cloud(fig.add_subplot(gs[0, 2]), amp, '(c) $P_0$, $P_2$ amplitudes', model_cov=amp_model_cov)
a = fig.add_subplot(gs[1, :2])
for m, off in (('G', -0.15), ('rec', 0.15)):
    s = np.var(cv[m]['y'][:, :N_DIR], axis=0, ddof=1)
    a.errorbar(np.arange(1, N_DIR + 1) + off, s, s * np.sqrt(2 / (len(B) - 1)), fmt='o', ms=3, color=MODEL_COLOR[m],
               label=MODEL_LABEL[m] + ' (out of sample)')
    a.plot(np.arange(1, N_DIR + 1), cv[m]['lam'][:N_DIR], ':', color=MODEL_COLOR[m], lw=0.8)
a.axhline(1, color='k', lw=0.6)
a.set(xlabel='direction (rank in half A)', ylabel='variance in half B', title='(d) cross-validated variance (dotted: in-sample)')
a.legend(fontsize=6.5)
a = fig.add_subplot(gs[1, 2])
for p, ls in zip(lead_patterns, ('-', '--')):
    for ell, c in ELL_COLOR.items():
        a.plot(k, p[block(ell)], color=c, ls=ls, lw=1)
a.axhline(0, color='k', lw=0.5); a.set(xlabel=KLABEL, title='(e) directions of (a)')
savefig(fig, 'clouds')

## 10. Template amplitudes (`templates`)
Wishart maximum-likelihood amplitudes $\theta$ of $C = C_{\rm G}+\sum_a\theta_aT_a$ with Fisher errors ($\theta=1$: prediction).
Takes ~1 min.

In [ ]:
FITS = [('SSC + disc', ('ssc', 'disc')), (r'+ tree $T_0$', ('ssc', 'disc', 'T0_tree')),
        (r'+ response $T_0$', ('ssc', 'disc', 'T0_response'))]
datasets = [(f'holi {t} {c}', holi(t, c)['V'], holi(t, c)['C']['G'], holi(t, c)['terms'])
            for t in ('LRG1', 'QSO') for c in ('NGC', 'SGC')]
datasets += [(f'Abacus LRG1 {c}', abacus('kernel', c)['V'], abacus('kernel', c)['C']['G'], abacus('kernel', c)['terms'])
             for c in ('NGC', 'SGC')]
fits = {}                     # [(dataset, fit)] -> (theta dict, sigma dict)
for name, V, CG, terms in datasets:
    for title, keys in FITS:
        if all(t in terms for t in keys):
            fits[name, title] = fit_templates(V, CG, {t: terms[t] for t in keys})
            print(name, title, {t: f'{fits[name, title][0][t]:.2f}+-{fits[name, title][1][t]:.2f}' for t in keys})

In [ ]:
TCOL = {'ssc': ORANGE, 'disc': AQUA, 'T0_tree': RED, 'T0_response': VIOLET}
TMARK = {'ssc': 'o', 'disc': 'o', 'T0_tree': 'D', 'T0_response': 's'}
XLIM = (-1, 3)
fig, ax = plt.subplots(1, 3, figsize=(WIDTH, 2.9), sharey=True)
names = [n for n, *_ in datasets]
ys = np.arange(len(names))[::-1]
for a, (title, keys) in zip(ax, FITS):
    for y, name in zip(ys, names):
        if (name, title) not in fits: continue
        th, sg = fits[name, title]
        for j, t in enumerate(keys):
            yy = y + (j - (len(keys) - 1) / 2) * 0.22
            x = np.clip(th[t], XLIM[0] + 0.1, XLIM[1] - 0.1)
            if x != th[t]:
                a.text(x, yy, f'{"<" if th[t] < 0 else ">"} {th[t]:.1f}', fontsize=6, color=TCOL[t], va='center')
            else:
                a.errorbar(th[t], yy, xerr=sg[t], fmt=TMARK[t], ms=3.2, color=TCOL[t], label=t if y == ys[0] else None)
    a.axvline(1, color='k', lw=0.6); a.axvline(0, color=LIGHT, lw=0.6)
    a.set(xlim=XLIM, xlabel=r'amplitude $\theta$', title=title); a.grid(axis='y', visible=False)
ax[0].set_yticks(ys, names)
handles = {l: h for a in ax for h, l in zip(*a.get_legend_handles_labels())}
fig.legend(handles.values(), handles.keys(), loc='upper center', ncol=4, bbox_to_anchor=(0.55, 1.06))
fig.tight_layout()
savefig(fig, 'templates')

## 11. Bin width (`binning`)
Same mocks at $\Delta k = 0.005, 0.01, 0.02$. A Gaussian-like model error gives a ratio independent of $\Delta k$; a
missing smooth non-Gaussian term grows $\propto\Delta k$; a low-rank term keeps the same leading eigenvalue.

In [ ]:
REBIN = (1, 2, 4)
DK = 0.005 * np.array(REBIN)
bw_trace, bw_diag = {}, {}       # [(cap, model)] -> per bin width; diag: (bin width, ell) mean ratio at 0.1<k<0.3
for cap in ('NGC', 'SGC'):
    for m in ('G_local', 'G', 'rec'):
        tr, dg = [], []
        for f in REBIN:
            d = holi('LRG1', cap, f)
            S, C, k = np.cov(d['V'].T), d['C'][m], d['k']
            tr.append(np.trace(np.linalg.solve(C, S)) / len(S))
            sel = np.flatnonzero((k > 0.1) & (k < 0.3))
            dg.append([np.mean(np.diag(S)[sel + j * len(k)] / np.diag(C)[sel + j * len(k)]) for j in range(3)])
        bw_trace[cap, m], bw_diag[cap, m] = np.array(tr), np.array(dg)
bw_lead = {}                     # [rebin] -> (k, leading direction pattern, eigenvalue), Gaussian model, NGC
for f in REBIN:
    d = holi('LRG1', 'NGC', f)
    W, Wi = whiten(d['V'], d['C']['G'])
    lam, U = np.linalg.eigh(W)
    p = np.linalg.solve(Wi, U[:, -1]) / np.sqrt(np.diag(d['C']['G']))
    bw_lead[f] = d['k'], p / p[np.argmax(np.abs(p))], lam[-1]

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(WIDTH, 2.35), gridspec_kw=dict(width_ratios=[1, 1, 1.3]))
for cap, ls, mfc in (('NGC', '-', None), ('SGC', '--', 'white')):
    for m in ('G_local', 'G', 'rec'):
        ax[0].plot(DK, bw_trace[cap, m], 'o', ls=ls, color=MODEL_COLOR[m], ms=3.5, mfc=mfc, label=MODEL_LABEL[m] if cap == 'NGC' else None)
        if m != 'G_local':
            for j, (ell, c) in enumerate(ELL_COLOR.items()):
                ax[1].plot(DK, bw_diag[cap, m][:, j], 'o', ls=ls if m == 'G' else ':', color=c, ms=3, mfc=mfc,
                           label=f'$\\ell={ell}$' if (cap, m) == ('NGC', 'G') else None)
ax[0].set(ylabel=r'${\rm tr}(C^{-1}S)/n$', title='(a) whitened trace')
ax[1].set(ylabel=r'$\langle\hat\sigma^2/\sigma^2_{\rm model}\rangle$', title='(b) diagonal, $0.1<k<0.3$')
ax[1].legend(fontsize=6.3, ncol=3)
for a in ax[:2]:
    a.set(xlabel=r'$\Delta k$', xticks=DK, xlim=(0.003, 0.022)); a.axhline(1, color='k', lw=0.6)
for f, lw in zip(REBIN, (0.8, 1.1, 1.6)):
    k, p, lam = bw_lead[f]
    for j, (ell, c) in enumerate(ELL_COLOR.items()):
        ax[2].plot(k, p[j * len(k):(j + 1) * len(k)], color=c, lw=lw, label=f'$\\Delta k={0.005*f:.3f}$, $\\lambda_1={lam:.1f}$' if ell == 0 else None)
ax[2].set(xlabel=KLABEL, ylabel=r'$v_1/\max|v_1|$', title='(c) leading direction'); ax[2].legend(fontsize=6)
fig.legend(*ax[0].get_legend_handles_labels(), loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.07), fontsize=7)
fig.tight_layout()
savefig(fig, 'binning')

## 12. Fibre assignment (`fiber`)
25 AbacusSummit mocks, complete vs altmtl (same initial conditions), each against its own Gaussian model.
(a) per-mock $\chi^2/n$; (b) paired double ratio of mock/model variance, altmtl over complete, per $k$ range.

In [ ]:
N_BOOT, KRANGES = 2000, ((0.02, 0.1), (0.1, 0.2), (0.2, 0.3))
rng = np.random.default_rng(3)
fib_chi2, fib_ratio = {}, {}
for cap in ('NGC', 'SGC'):
    c, a_ = abacus('complete', cap), abacus('altmtl', cap)
    assert (c['ids'] == a_['ids']).all()
    N, n = c['V'].shape
    xc = chi2(c['V'], c['C']['G_local']) / (n * (N - 1) / N)
    xa = chi2(a_['V'], a_['C']['G_local']) / (n * (N - 1) / N)
    fib_chi2[cap] = xc, xa
    print(f'{cap}: <chi2/n> altmtl - complete = {np.mean(xa - xc):+.3f} +- {np.std(xa - xc, ddof=1) / np.sqrt(N):.3f}, corr {np.corrcoef(xc, xa)[0, 1]:.2f}')
    model_ratio = np.diag(a_['C']['G_local']) / np.diag(c['C']['G_local'])
    k = c['k']
    def double_ratio(s):
        lr = np.log(a_['V'][s].var(0, ddof=1) / c['V'][s].var(0, ddof=1) / model_ratio)
        return np.array([[np.exp(lr[block(ell, len(k))][(k >= lo) & (k < hi)].mean()) for lo, hi in KRANGES] for ell in ELLS])
    r = double_ratio(np.arange(N))
    boot = np.array([double_ratio(rng.integers(0, N, N)) for _ in range(N_BOOT)])
    fib_ratio[cap] = r, *np.percentile(boot, [16, 84], axis=0)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(WIDTH, 2.5), gridspec_kw=dict(width_ratios=[1, 1.7]))
for cap, c in (('NGC', BLUE), ('SGC', ORANGE)):
    xc, xa = fib_chi2[cap]
    ax[0].scatter(xc, xa, s=12, color=c, edgecolor='w', lw=0.5, label=cap)
ax[0].plot((0.7, 1.75), (0.7, 1.75), 'k--', lw=0.7)
ax[0].set(xlim=(0.7, 1.75), ylim=(0.7, 1.75), aspect='equal', xlabel=r'$\chi^2_i/n$, complete', ylabel=r'$\chi^2_i/n$, altmtl', title='(a) per-mock $\\chi^2$')
ax[0].legend()
for cap, mk, off in (('NGC', 'o', -0.06), ('SGC', 's', 0.06)):
    r, lo, hi = fib_ratio[cap]
    for j, (ell, c) in enumerate(ELL_COLOR.items()):
        x = np.arange(3) + (j - 1) * 0.22 + off
        ax[1].errorbar(x, r[j], [r[j] - lo[j], hi[j] - r[j]], fmt=mk, color=c, ms=3.2, mfc=c if cap == 'NGC' else 'w',
                       label=f'$\\ell={ell}$' if cap == 'NGC' else None)
ax[1].axhline(1, color='k', lw=0.6)
ax[1].set(ylim=(0.8, 1.2), ylabel='double ratio', title='(b) altmtl / complete (filled NGC, open SGC)')
ax[1].set_xticks(range(3), [f'${lo}<k<{hi}$' for lo, hi in KRANGES]); ax[1].legend(ncol=3)
fig.tight_layout()
savefig(fig, 'fiber')

## 13. Parameter errors (`params`)
Each mock fitted (GLS, model covariance) for $A$ (all multipoles), $A_2$ (extra quadrupole), $\alpha$ (dilation) and a
monopole constant; variance over the mocks / predicted variance, vs $k_{\max}$.

In [ ]:
PARAM_NAMES = (r'$A$', r'$A_2$', r'$\alpha$', r'$N_{\rm SN}$')
def param_ratios(V, C, k, kmax):
    nb, Pm = len(k), V.mean(0)
    P0, P2, P4 = Pm[:nb], Pm[nb:2 * nb], Pm[2 * nb:]
    dP = lambda P: -np.gradient(P, np.log(k))
    z = np.zeros(nb)
    T = np.stack([np.r_[P0, P2, P4], np.r_[z, P2, z], np.r_[dP(P0), dP(P2), dP(P4)], np.r_[np.ones(nb), z, z]], 1)
    I = kidx(k, kmax)
    T, Ci = T[I], np.linalg.inv(C[np.ix_(I, I)])
    Fi = np.linalg.inv(T.T @ Ci @ T)
    est = (Fi @ T.T @ Ci @ (V[:, I] - V[:, I].mean(0)).T).T
    return est.var(0, ddof=1) / np.diag(Fi)

KMAX_P = np.arange(0.1, 0.301, 0.025)
par = {}                         # [(tracer, cap, model)] -> (n_kmax, 4)
for tracer, cap in (('LRG1', 'NGC'), ('LRG1', 'SGC'), ('QSO', 'NGC')):
    d = holi(tracer, cap)
    for m in ('G_local', 'G', 'rec'):
        par[tracer, cap, m] = np.array([param_ratios(d['V'], d['C'][m], d['k'], km) for km in KMAX_P])

In [ ]:
rows = (('LRG1', 'NGC'), ('LRG1', 'SGC'), ('QSO', 'NGC'))
fig, ax = plt.subplots(len(rows), 4, figsize=(WIDTH, 4.6), sharex=True, sharey=True)
for i, (tracer, cap) in enumerate(rows):
    N = len(holi(tracer, cap)['V'])
    for j in range(4):
        a = ax[i, j]
        for m in ('G_local', 'G', 'rec'):
            a.plot(KMAX_P, par[tracer, cap, m][:, j], 'o-', ms=2.5, color=MODEL_COLOR[m], label=MODEL_LABEL[m])
        a.axhspan(1 - np.sqrt(2 / (N - 1)), 1 + np.sqrt(2 / (N - 1)), color=LIGHT, lw=0)
        a.axhline(1, color='k', lw=0.6)
        if i == 0: a.set_title(PARAM_NAMES[j])
        if j == 0: a.set_ylabel(f'{tracer} {cap}\n' + r'$\hat\sigma^2/\sigma^2_{\rm model}$')
        if i == len(rows) - 1: a.set_xlabel(r'$k_{\max}$')
ax[0, 0].set(yscale='log', ylim=(0.75, 2.0)); ax[0, 0].set_yticks([0.8, 1, 1.25, 1.5, 2], ['0.8', '1', '1.25', '1.5', '2'])
ax[0, 0].minorticks_off()
fig.legend(*ax[0, 0].get_legend_handles_labels(), loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.04), fontsize=7)
fig.tight_layout()
savefig(fig, 'params')

## 14. Radial long modes from the mocks' $n(z)$ (`nz`)
(a) $n(z)$ of each LRG1 NGC mock / median (red: excluded mocks); (b) rms of slab counts (Poisson removed) over linear
theory $(b_1+f)\sigma_{\rm slab}(L)$ vs slab thickness; filled: without the excluded mocks, open: all.

In [ ]:
from scipy.integrate import quad
OMEGA_M = 0.3153
chi = lambda z: 2997.92458 * quad(lambda x: 1 / np.sqrt(OMEGA_M * (1 + x)**3 + 1 - OMEGA_M), 0, z)[0]

def slab_sigma(L, area, kl, P):
    q = np.linspace(1e-4, 2.0, 200000)
    return np.sqrt(np.trapezoid(np.interp(q, kl, P) * np.sinc(q * L / 2 / np.pi)**2, q) / np.pi / area)

z = np.load(FILES['nz'].format(tracer='LRG1'))
zc = 0.5 * (z['edges'][1:] + z['edges'][:-1])
nz_dev = z['NGC/NZ'] / np.median(z['NGC/NZ'], axis=0) - 1
nz_bad = np.isin(z['NGC/mocks'], EXCLUDED['LRG1'])
nz_ratio = {}                    # [(tracer, cap, clean)] -> (thickness, ratio, error)
for tracer in ('LRG1', 'QSO'):
    z = np.load(FILES['nz'].format(tracer=tracer))
    p = holi(tracer, 'NGC')['params']; kl, P = holi(tracer, 'NGC')['plin']
    c0, c1 = chi(z['edges'][0]), chi(z['edges'][-1])
    for cap in ('NGC', 'SGC'):
        area = float(z[f'{cap}/area']) * (c1**3 - c0**3) / 3 / (c1 - c0)       # sr x <r^2>
        for clean in (False, True):
            M = z[f'{cap}/NZ'][~np.isin(z[f'{cap}/mocks'], EXCLUDED[tracer])] if clean else z[f'{cap}/NZ']
            L, y, e = [], [], []
            for nmerge in (1, 2, 5, 10, 20):
                mg = M.reshape(len(M), nmerge, -1).sum(2)
                rms = np.sqrt(np.maximum((mg.std(0, ddof=1) / mg.mean(0))**2 - 1 / mg.mean(0), 0)).mean()
                th = (p['b1'] + p['f']) * slab_sigma((c1 - c0) / nmerge, area, kl, P)
                L.append((c1 - c0) / nmerge); y.append(rms / th); e.append(rms / th * np.sqrt(nmerge / (2 * (len(M) - 1))))
            nz_ratio[tracer, cap, clean] = np.array(L), np.array(y), np.array(e)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(WIDTH, 2.5), gridspec_kw=dict(width_ratios=[1.1, 1]))
for row in nz_dev[~nz_bad][:300]:
    ax[0].plot(zc, 100 * row, color=GREY, lw=0.3, alpha=0.4)
for row in nz_dev[nz_bad]:
    ax[0].plot(zc, 100 * row, color=RED, lw=0.9)
ax[0].set(xlabel='$z$', ylabel=r'$n(z)/{\rm median}-1$ [%]', title=f'(a) LRG1 NGC: {nz_bad.sum()} defective mocks')
for (tracer, cap, clean), (L, y, e) in nz_ratio.items():
    if not clean and not EXCLUDED[tracer]: continue
    c = BLUE if tracer == 'LRG1' else ORANGE
    ax[1].errorbar(L, y, e, fmt='o-' if cap == 'NGC' else 'o--', color=c, ms=3, mfc=c if clean else 'w',
                   label=f'{tracer} {cap}' if clean else None)
ax[1].axhline(1, color='k', lw=0.6)
ax[1].set(xscale='log', xlabel=r'slab thickness [$h^{-1}$Mpc]', ylabel=r'$\sigma_{\rm mocks}/\sigma_{\rm linear}$', title='(b) radial modes')
ax[1].legend(fontsize=6.5, ncol=2)
fig.tight_layout()
savefig(fig, 'nz')